In [27]:
# Zachary Katz
# zachary_katz@mines.edu
# 07 May 2025
# Gigi Albers
# Imports

%load_ext autoreload
%autoreload 2

import util.plotting_helpers as plothelp
import matplotlib.pyplot as plt
import earthaccess
import xarray as xr
import numpy as np
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from pyproj import Transformer
from scipy.interpolate import griddata




auth = earthaccess.login(strategy="netrc")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


ModuleNotFoundError: No module named 'scipy'

In [28]:
print("Enter BEFORE SWOT granule details:")
results = earthaccess.search_data(
        short_name = "SWOT_L2_HR_Raster_D",
        granule_name= input("Granule name (e.g., SWOT_L2_HR_Raster_250m_UTM54U_N_x_x_x_...): "),
        temporal= (
            input("Start date (YYYY-MM-DD, e.g., 2025-06-02): "),
            input("End date (YYYY-MM-DD, same as start for single day): ")
        ),
)

print(f"Number of granules found: {len(results)}")
if len(results) == 0:
    print("No granules found! Check your search parameters.")
else:
    print("Granules found. Proceeding to open...")

print("\nEnter bounding box coordinates [min_lon, max_lon, min_lat, max_lat]:")

swot_ds_before = xr.open_dataset(earthaccess.open(results)[0], engine="h5netcdf")

print("Enter AFTER SWOT granule details:")
results = earthaccess.search_data(
        short_name = "SWOT_L2_HR_Raster_D",
        granule_name= input("Granule name (e.g., SWOT_L2_HR_Raster_250m_UTM54U_N_x_x_x_...): "),
        temporal= (
            input("Start date (YYYY-MM-DD, e.g., 2025-06-02): "),
            input("End date (YYYY-MM-DD, same as start for single day: ")
        ),
)

print(f"Number of granules found: {len(results)}")
if len(results) == 0:
    print("No granules found! Check your search parameters.")
else:
    print("Granules found. Proceeding to open...")

print("\nEnter bounding box coordinates [min_lon, max_lon, min_lat, max_lat]:")
bbox = [
    float(input("Min longitude: ")),
    float(input("Max longitude: ")),
    float(input("Min latitude: ")),
    float(input("Max latitude: "))
]

swot_ds_after = xr.open_dataset(earthaccess.open(results)[0], engine="h5netcdf")

Enter BEFORE SWOT granule details:


Granule name (e.g., SWOT_L2_HR_Raster_250m_UTM54U_N_x_x_x_...):  SWOT_L2_HR_Raster_100m_UTM43P_N_x_x_x_033_148_067F_20250524T040104_20250524T040120_PID0_01.nc
Start date (YYYY-MM-DD, e.g., 2025-06-02):  2025-05-24
End date (YYYY-MM-DD, same as start for single day):  2025-05-24


Number of granules found: 1
Granules found. Proceeding to open...

Enter bounding box coordinates [min_lon, max_lon, min_lat, max_lat]:


QUEUEING TASKS | : 100%|██████████| 1/1 [00:00<00:00, 1254.65it/s]
PROCESSING TASKS | : 100%|██████████| 1/1 [00:00<00:00, 26.76it/s]
COLLECTING RESULTS | : 100%|██████████| 1/1 [00:00<00:00, 16194.22it/s]


Enter AFTER SWOT granule details:


Granule name (e.g., SWOT_L2_HR_Raster_250m_UTM54U_N_x_x_x_...):  SWOT_L2_HR_Raster_250m_UTM43P_N_x_x_x_034_148_067F_20250614T004608_20250614T004624_PID0_01.nc
Start date (YYYY-MM-DD, e.g., 2025-06-02):  2025-06-14
End date (YYYY-MM-DD, same as start for single day:  2025-06-14


Number of granules found: 1
Granules found. Proceeding to open...

Enter bounding box coordinates [min_lon, max_lon, min_lat, max_lat]:


Min longitude:  11.29629
Max longitude:  13.84334
Min latitude:  73.65126
Max latitude:  76.36676


QUEUEING TASKS | : 100%|██████████| 1/1 [00:00<00:00, 2107.69it/s]
PROCESSING TASKS | : 100%|██████████| 1/1 [00:00<00:00, 18.86it/s]
COLLECTING RESULTS | : 100%|██████████| 1/1 [00:00<00:00, 9467.95it/s]


In [29]:

def resample_to_common_grid(X1, Y1, data1, X2, Y2, data2):
    """Resample both datasets to a common lat/lon grid using interpolation."""
    # Define common bounds
    min_lon = min(X1.min(), X2.min())
    max_lon = max(X1.max(), X2.max())
    min_lat = min(Y1.min(), Y2.min())
    max_lat = max(Y1.max(), Y2.max())
    
    # Create a common grid
    xi = np.linspace(min_lon, max_lon, 500)
    yi = np.linspace(min_lat, max_lat, 500)
    xi, yi = np.meshgrid(xi, yi)
    
    # Flatten for interpolation
    points1 = np.column_stack((X1.ravel(), Y1.ravel()))
    points2 = np.column_stack((X2.ravel(), Y2.ravel()))
    
    # Interpolate to common grid
    data1_interp = griddata(points1, data1.ravel(), (xi, yi), method='linear')
    data2_interp = griddata(points2, data2.ravel(), (xi, yi), method='linear')
    
    return xi, yi, data1_interp, data2_interp


In [ ]:
fig = plt.figure(figsize=(30/2.54, 15/2.54))  # Width, Height in inches (30cm wide, 15cm tall)
gs = fig.add_gridspec(1, 3, width_ratios=[1, 1, 0.05], wspace=0.7)  # Two plots, one colorbar

# Create axes for the plots
ax1 = fig.add_subplot(gs[0, 0], projection=ccrs.PlateCarree())
ax2 = fig.add_subplot(gs[0, 1], projection=ccrs.PlateCarree())
cax = fig.add_subplot(gs[0, 2])  # Colorbar axis

# Function to plot SWOT data on a given axis
def plot_swot_data(ax, ds, title):
    # Get data
    wse = ds["wse"] + ds["height_cor_xover"]
    
    # Convert UTM to lat/lon
    utm_zone = ds.utm_zone_num
    utm_crs = ccrs.UTM(zone=utm_zone, southern_hemisphere=False)
    transformer = Transformer.from_crs(utm_crs, ccrs.PlateCarree(), always_xy=True)
    x_utm, y_utm = wse["x"].values, wse["y"].values
    X_utm, Y_utm = np.meshgrid(x_utm, y_utm)
    X_lon, Y_lat = transformer.transform(X_utm, Y_utm)
    
    # Plot data
    mesh = ax.pcolormesh(
        X_lon,
        Y_lat,
        wse.values,
        transform=ccrs.PlateCarree(),
        cmap="viridis",
        vmin=0,
        vmax=10,
    )
    
    # Add map features
    ax.gridlines(draw_labels=True)
    ax.add_feature(cfeature.LAND, facecolor='lightgray')
    ax.add_feature(cfeature.OCEAN, facecolor='lightblue')
    ax.coastlines(resolution='10m', linewidth=1)
    ax.set_title(title, fontsize=14)
    
    return mesh

# Plot both datasets
mesh1 = plot_swot_data(ax1, swot_ds_before, input("Before Event (Format: Specific location, mm/dd/yyyy): "))
mesh2 = plot_swot_data(ax2, swot_ds_after, input("After Event (Format: Specific location, mm/dd/yyyy): "))

# Add single colorbar
cbar = fig.colorbar(mesh1, cax=cax, orientation='vertical', location ='right', pad=0.5)
cbar.set_label(
    "Water Surface Elevation [m]",
    fontsize=12,
    labelpad=15,  # Space between colorbar and label
    ha='center',  # Horizontal alignment
    va='bottom',  # Vertical alignment
    rotation=90    # 0=horizontal, 90=vertical
)


fig.text(
    x=0.45,  # Left alignment (0=far left, 1=far right)
    y=0.90,   # Vertical position (1=top of figure)
    s=input("Main location: "),  # Your custom title text
    fontsize=20,
    color='black',
    va='top', # Vertical alignment
    ha='center'   # Horizontal alignment
)

Before Event (Format: Specific location, mm/dd/yyyy):  Mangaluru, 05/24/2025
After Event (Format: Specific location, mm/dd/yyyy):  Mangaluru, 06/14/2025


In [26]:
def plot_difference(ax, X_lon, Y_lat, difference, title):
    """Plot difference data on a given axis"""
    mesh = ax.pcolormesh(
        X_lon,
        Y_lat,
        difference,
        transform=ccrs.PlateCarree(),
        cmap="coolwarm",
        vmin=-5,
        vmax=5,
    )
    
    # Add map features
    ax.gridlines(draw_labels=True)
    ax.add_feature(cfeature.LAND, facecolor='lightgray')
    ax.add_feature(cfeature.OCEAN, facecolor='lightblue')
    ax.coastlines(resolution='10m', linewidth=1)
    ax.set_title(title, fontsize=14)
    
    return mesh

# Prepare data from both datasets
# BEFORE dataset
before_wse = swot_ds_before["wse"].values + swot_ds_before["height_cor_xover"].values
before_x = swot_ds_before["x"].values
before_y = swot_ds_before["y"].values
X1_utm, Y1_utm = np.meshgrid(before_x, before_y)

# AFTER dataset
after_wse = swot_ds_after["wse"].values + swot_ds_after["height_cor_xover"].values
after_x = swot_ds_after["x"].values
after_y = swot_ds_after["y"].values
X2_utm, Y2_utm = np.meshgrid(after_x, after_y)

# Set UTM zone (replace 54 with correct zone if needed)
utm_zone = 54
transformer = Transformer.from_crs(
    f"+proj=utm +zone={utm_zone} +north +ellps=WGS84 +datum=WGS84 +units=m +no_defs",
    "EPSG:4326",
    always_xy=True
)

# Convert UTM to lat/lon
X1, Y1 = transformer.transform(X1_utm, Y1_utm)
X2, Y2 = transformer.transform(X2_utm, Y2_utm)

# Assign data for later steps
before_data = before_wse
after_data = after_wse


# Resample to common grid
X_common, Y_common, before_interp, after_interp = resample_to_common_grid(
    X1, Y1, before_data, X2, Y2, after_data)

# Calculate difference
difference = after_interp - before_interp

# Plot data
before_title = input("Before Event (Format: Specific location, mm/dd/yyyy): ")
after_title = input("After Event (Format: Specific location, mm/dd/yyyy): ")

mesh1 = plot_swot_data(ax1, X_common, Y_common, before_interp, before_title)
mesh2 = plot_swot_data(ax2, X_common, Y_common, after_interp, after_title)
mesh3 = plot_difference(ax3, X_common, Y_common, difference, "Difference (After - Before)")

# Add single colorbar for difference plot
cbar = fig.colorbar(mesh3, cax=cax, orientation='vertical', location='right', pad=0.5)
cbar.set_label(
    "Water Surface Elevation Difference [m]",
    fontsize=12,
    labelpad=15,
    ha='center',
    va='bottom',
    rotation=90
)

fig.text(
    x=0.5,
    y=0.90,
    s=input("Main location: "),
    fontsize=20,
    color='black',
    va='top',
    ha='center'
)

NameError: name 'resample_to_common_grid' is not defined

In [ ]:
def plot_small_inset(fig, position, bbox):
    ax_inset = fig.add_axes(position, projection=ccrs.PlateCarree())
    
    # Find a new wide set of longitudes and latitudes that clearly indicate where you are
    ax_inset.set_extent([45,55,70,75], crs=ccrs.PlateCarree()) #[longitude_min, longitude_max, latitude_min, latitude_max], [-20, 20, 0, 30]

                    
    # Add features
    ax_inset.add_feature(cfeature.LAND, facecolor='lightgray')
    ax_inset.add_feature(cfeature.OCEAN, facecolor='lightblue')
    ax_inset.coastlines(resolution='50m', linewidth=0.5)
    
    # Add red rectangle showing specific area 
    rect = plt.Rectangle(
    (bbox[1], bbox[0]), 
    bbox[2]-bbox[0], 
    bbox[3]-bbox[1],
    fill=False, color='red', linewidth=1,
    transform=ccrs.PlateCarree()
)
    ax_inset.add_patch(rect)
    
    ax_inset.set_title("Location", fontsize=7)
    return ax_inset

inset = plot_small_inset(
    fig=fig,
    position=[0.95, 0.5, 0.25, 0.25],  # [left, bottom, width, height]
    bbox= bbox #[3.35058, 6.26802, 4.6684, 7.57602]
)

plt.show()